# 01 Prepare coolers

Locate original cooler files, downsample them @ 20Kb resolution, balance, and move to mcool files.

REDO: use the cooler_repo from 6575 to locate coolers; copy or downsample all coolers to here. This means that I will copy dSororin_G2, WT_G2_BR1n2, WT_G2_BR3_ds_1n2, Prometa, and downsample WT_G2 and WT_G1. I will specifically do this @ 20Kb resolution and create mcool files here.

In [1]:
from pathlib import Path
import cooler
import cooltools
from multiprocessing import Pool
from tqdm import tqdm

In [16]:
CORES = 8
RESOLUTIONS = [10_000, 20_000]
BASE_RES = RESOLUTIONS[0]

## Biological conditions

### Locate files

In [17]:
cooler_repo = Path('/groups/gerlich/experiments/Experiments_006500/006575/coolers_repo/')

In [18]:
conditions = ('WT_G2', 'WT_G1', 'dSororin_G2', 'Prometa', 'WT_G2_BR1n2', 'WT_G2_BR3_ds_1n2')
cond_folders = {cond: cooler_repo / f"{'_'.join(cond.split('_')[:2])}"
                for cond in conditions}

In [19]:
cond_folders

{'WT_G2': PosixPath('/groups/gerlich/experiments/Experiments_006500/006575/coolers_repo/WT_G2'),
 'WT_G1': PosixPath('/groups/gerlich/experiments/Experiments_006500/006575/coolers_repo/WT_G1'),
 'dSororin_G2': PosixPath('/groups/gerlich/experiments/Experiments_006500/006575/coolers_repo/dSororin_G2'),
 'Prometa': PosixPath('/groups/gerlich/experiments/Experiments_006500/006575/coolers_repo/Prometa'),
 'WT_G2_BR1n2': PosixPath('/groups/gerlich/experiments/Experiments_006500/006575/coolers_repo/WT_G2'),
 'WT_G2_BR3_ds_1n2': PosixPath('/groups/gerlich/experiments/Experiments_006500/006575/coolers_repo/WT_G2')}

In [21]:
clr_paths = {cond: str(cond_folders[cond] / f"{cond}.all.mcool")
             for cond in conditions}

In [23]:
for cond, path in clr_paths.items():
    src_path = path + f"::/resolutions/{BASE_RES}"
    print(cond, f"{cooler.Cooler(src_path).info['sum']:,d}", sep='\t')

WT_G2	1,685,756,604
WT_G1	1,052,824,740
dSororin_G2	499,168,270
Prometa	99,993,251
WT_G2_BR1n2	333,248,942
WT_G2_BR3_ds_1n2	333,256,757


We will leave Prometa as is (we don't see TADs and don't expect TADs anyway). We have already downsampled and matched bioreps of WT G2. Meaning I will downsample only WT_G2 and WT_G1 to dSororin_G2.

### Copy or downsample and balance coolers

In [24]:
target_cond = 'dSororin_G2'
target_n = cooler.Cooler(str(clr_paths[target_cond]) + f"::/resolutions/{BASE_RES}").info['sum']

In [25]:
target_n

499168270

In [31]:
for cond, path in tqdm(clr_paths.items()):
    src_path = str(path)
    dest_path = f"./{cond}.mcool"

    if cooler.Cooler(src_path + f"::/resolutions/{BASE_RES}").info['sum'] <= target_n:

        for res in RESOLUTIONS:
            print(f'copying {cond} @ {res}')
            src_uri = src_path + f"::/resolutions/{res}"
            dest_uri = dest_path + f"::/resolutions/{res}"
            cooler.fileops.cp(src_uri, dest_uri)
    else:
        
        with Pool(CORES) as p:
            print(f'downsampling {cond} @ {BASE_RES}')
            src_uri = src_path + f"::/resolutions/{BASE_RES}"
            base_dest_uri = dest_path + f"::/resolutions/{BASE_RES}"
            cooltools.sample.__wrapped__(cooler.Cooler(src_uri),
                                         base_dest_uri,
                                         count=target_n,
                                         map_functor=p.map)
        for res in RESOLUTIONS:
            dest_uri = dest_path + f"::/resolutions/{res}"
            if res > BASE_RES:
                print(f'coarsening {cond} @ {res}')
                cooler.coarsen_cooler(base_dest_uri,
                                      dest_uri,
                                      factor=res // BASE_RES,  # Assuming that resolutions are factors of the base resolution
                                      chunksize=10_000_000,
                                      nproc=CORES)
            with Pool(CORES) as p:
                print(f'Balancing {cond} @ {res}')
                cooler.balance_cooler(cooler.Cooler(dest_uri),
                                      ignore_diags=2,
                                      mad_max=5,
                                      map=p.map,
                                      store=True)

  0%|          | 0/6 [00:00<?, ?it/s]

downsampling WT_G2 @ 10000
Balancing WT_G2 @ 10000
coarsening WT_G2 @ 20000
Balancing WT_G2 @ 20000


 17%|█▋        | 1/6 [07:02<35:13, 422.67s/it]

downsampling WT_G1 @ 10000
Balancing WT_G1 @ 10000
coarsening WT_G1 @ 20000
Balancing WT_G1 @ 20000


 33%|███▎      | 2/6 [18:52<39:26, 591.63s/it]

...

 67%|██████▋   | 4/6 [18:54<06:30, 195.31s/it]

copying WT_G2_BR1n2 @ 10000
copying WT_G2_BR1n2 @ 20000


 83%|████████▎ | 5/6 [18:56<02:05, 125.34s/it]

copying WT_G2_BR3_ds_1n2 @ 10000
copying WT_G2_BR3_ds_1n2 @ 20000


100%|██████████| 6/6 [18:57<00:00, 189.56s/it]


### Verify files

In [33]:
for cond in clr_paths:
    mcool_path = f"./{cond}.mcool"
    print(cooler.fileops.list_coolers(mcool_path))
    for res in RESOLUTIONS:
        dest_path = mcool_path + f"::/resolutions/{res}"
        print(cooler.Cooler(dest_path).bins()[:10])
        print(cooler.Cooler(dest_path).info['sum'])

['/resolutions/10000', '/resolutions/20000']
  chrom  start     end    weight
0  chr1      0   10000       NaN
1  chr1  10000   20000  0.067290
2  chr1  20000   30000       NaN
3  chr1  30000   40000       NaN
4  chr1  40000   50000       NaN
5  chr1  50000   60000  0.054097
6  chr1  60000   70000  0.049096
7  chr1  70000   80000  0.058306
8  chr1  80000   90000  0.029427
9  chr1  90000  100000       NaN
499167425
  chrom   start     end    weight
0  chr1       0   20000       NaN
1  chr1   20000   40000       NaN
2  chr1   40000   60000       NaN
3  chr1   60000   80000  0.039066
4  chr1   80000  100000  0.033415
5  chr1  100000  120000  0.040680
6  chr1  120000  140000       NaN
7  chr1  140000  160000       NaN
8  chr1  160000  180000       NaN
9  chr1  180000  200000       NaN
499167425
['/resolutions/10000', '/resolutions/20000']
  chrom  start     end    weight
0  chr1      0   10000       NaN
1  chr1  10000   20000       NaN
2  chr1  20000   30000       NaN
3  chr1  30000   4000

mcool files exist and are balanced at both 10Kb and 20Kb resolutions.

## REMOVE BEFORE PUBLICATION (legacy code)

## G2 WT replicates

### Locate replicates


Data was published in Mitter2020.

In [3]:
g2_reps = input_files = {'BR1': '/groups/gerlich/archive/experiments/Experiments_004600/004661/Sequencing_data/cooler/exp4605/GCCAAT.all.1000.cool',
                         'BR2': '/groups/gerlich/archive/experiments/Experiments_004600/004661/Sequencing_data/cooler/exp4615/GTGAAA.all.1000.cool',
                         'BR3': '/groups/gerlich/archive/experiments/Experiments_004800/004812/Sequencing_data/Pooled_FC_1_2_3_4/cooler/G2.fc_1_2_3_4.all.1000.cool'}

In [8]:
for rep, clr_path in g2_reps.items():
    print(rep, f"{cooler.Cooler(clr_path).info['sum']:,d}")

BR1 219,410,439
BR2 113,838,503
BR3 1,352,507,662


We will merge BR1 and BR2 together and downsample BR3 to match the number of contacts in BR1n2.

### Merge BR1 and BR2

In [11]:
%%time
br1n2_path = './G2_WT.BR1n2.1000.cool'
cooler.merge_coolers(br1n2_path, [g2_reps['BR1'], g2_reps['BR2']], int(1e7))

CPU times: user 2min 17s, sys: 41.7 s, total: 2min 59s
Wall time: 3min 1s


In [12]:
n_merged = cooler.Cooler(br1n2_path).info['sum'] 
print(f"{n_merged:,d}")

333,248,942


### Downsample BR3 to match BR1n2

In [17]:
%%time
br3_path = './G2_WT.BR3.1000.cool'
with Pool(CORES) as p:
    cooltools.sample.__wrapped__(cooler.Cooler(g2_reps['BR3']), br3_path, count=n_merged, map_functor=p.map)

CPU times: user 45.5 s, sys: 12.6 s, total: 58.1 s
Wall time: 1min 1s


In [21]:
print(f"{cooler.Cooler(br3_path).info['sum']:,d}")

333,267,908


### Zoomify BR1n2 and BR3

In [19]:
for in_path, out_path in tqdm(zip((br1n2_path, br3_path),
                                  ('./G2_WT.BR1n2.mcool', './G2_WT.BR3.mcool'))):
    cooler.zoomify_cooler(in_path, out_path, [20_000,], int(1e7), nproc=8)
    with Pool(CORES) as p:
        cooler.balance_cooler(cooler.Cooler(out_path + "::/resolutions/20000"), ignore_diags=2, mad_max=5, map=p.map, store=True)

2it [04:05, 122.86s/it]


### Verify files

In [23]:
cooler.fileops.list_coolers('./G2_WT.BR1n2.mcool')

['/resolutions/1000', '/resolutions/20000']

In [34]:
cooler.Cooler('./G2_WT.BR1n2.mcool::/resolutions/20000').bins()[:10]

,chrom,start,end,weight
0,chr1,0,20000,NaN
1,chr1,20000,40000,NaN
2,chr1,40000,60000,NaN
3,chr1,60000,80000,0.045798
4,chr1,80000,100000,0.041405
5,chr1,100000,120000,NaN
6,chr1,120000,140000,NaN
7,chr1,140000,160000,NaN
8,chr1,160000,180000,NaN
9,chr1,180000,200000,NaN


In [24]:
cooler.fileops.list_coolers('./G2_WT.BR3.mcool')

['/resolutions/1000', '/resolutions/20000']

In [35]:
cooler.Cooler('./G2_WT.BR3.mcool::/resolutions/20000').bins()[:10]

,chrom,start,end,weight
0,chr1,0,20000,NaN
1,chr1,20000,40000,NaN
2,chr1,40000,60000,NaN
3,chr1,60000,80000,0.051844
4,chr1,80000,100000,0.039633
5,chr1,100000,120000,0.048172
6,chr1,120000,140000,NaN
7,chr1,140000,160000,NaN
8,chr1,160000,180000,NaN
9,chr1,180000,200000,NaN


mcool files are alright and 20Kb res coolers are balanced.

### Cleanup

In [37]:
Path(br1n2_path).unlink()
Path(br3_path).unlink()